# Training diagnosis triase (multi-tabel → 1 baris per sampel)

Pola umum soal multi-tabel, tidak terikat domain:
1. **Tabel utama** (`train`/`test`): 1 baris = 1 sampel yang diprediksi. Semua fitur akhirnya ditempel ke sini.
2. **Tabel 1-ke-1** (pasien): tinggal `merge`, bersihkan dulu.
3. **Tabel long per sampel** (vital, gejala): `pivot_table` jadi kolom (1 baris per sampel).
4. **Tabel riwayat per entitas** (riwayat medis): **filter waktu dulu** (hanya sebelum `prediction_time`), lalu hitung jumlah/jarak hari.
5. Model: LightGBM + StratifiedKFold (diulang), nilai OOF dengan **macro-F1** (kelas tidak seimbang), lalu atur bobot kelas di OOF.

Baris bertanda `# UBAH` = yang diganti kalau soalnya beda.

Jebakan data yang sudah ditangani: umur 999, income campur `USD 1,234` / `115.6k`, status nikah `M`/`Married`,
vital koma desimal (`28,02`) dan satuan beda (kPa vs mmHg, fraction vs %), nama gejala bersinonim, riwayat harus sebelum waktu prediksi.

**Perubahan dari versi sebelumnya**
- Path disesuaikan dengan struktur zip (`Triase/Triase`, `Riwayat Medis/Riwayat Medis`); zip diekstrak otomatis bila perlu.
- CV diulang `N_REPEAT` kali supaya OOF dan bobot kelas lebih stabil (kelas strep hanya 195 sampel).
- Aturan "naiknya cuma ~0,001, abaikan" sekarang benar-benar ada di kode, plus uji kestabilan bobot (setel di separuh data, nilai di separuh lain).
- Kolom konstan dibuang; bila LightGBM tidak terpasang, otomatis memakai `HistGradientBoostingClassifier`.

**Catatan penting tentang fitur teks (TF-IDF)**: teks gejala + keluhan menaikkan skor jauh, tetapi sumber sinyalnya adalah *urutan baris gejala*
dalam satu encounter (gejala pertama sangat berbeda antar kelas, dan pola yang sama ada di test), bukan teks keluhan. Ini sidik jari cara data
dihasilkan, bukan temuan klinis, jadi jangan diklaim sebagai pola medis di write-up. Teks **riwayat** (nama penyakit/obat) tidak dipakai (menurunkan CV).

In [1]:
import re, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:                      # tanpa LightGBM -> HistGradientBoosting (lihat bagian 7)
    HAS_LGB = False

ZIP_NAMA = "gdgoc-ugm-academic-hacker-2026.zip"
def cari_folder():
    """Folder data = yang berisi train.csv. Kalau hanya ada zip-nya, ekstrak ke ./data."""
    for c in [Path("../data"), Path("data"), Path(".")]:           # UBAH: tambah kandidat lain bila perlu
        if (c / "train.csv").exists():
            return c
    for z in [Path(ZIP_NAMA), Path("..") / ZIP_NAMA, Path("../data") / ZIP_NAMA, Path("data") / ZIP_NAMA]:
        if z.exists():
            zipfile.ZipFile(z).extractall("data"); return Path("data")
    raise FileNotFoundError("train.csv / zip tidak ditemukan; atur FOLDER manual")

FOLDER = cari_folder()
def sub(nama):
    """Subfolder triase/riwayat. Zip: 'Triase/Triase'; juga menerima versi diratakan atau 'triase'."""
    for c in [FOLDER / nama / nama, FOLDER / nama, FOLDER / nama.lower().replace(" ", "-")]:
        if c.is_dir():
            return c
    raise FileNotFoundError(f"subfolder '{nama}' tidak ditemukan di {FOLDER}")
TRIASE, RIWAYAT_DIR = sub("Triase"), sub("Riwayat Medis")

OUT = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)
KEY = "encounter_id"                # UBAH: kunci sampel ke tabel per-kunjungan
PASIEN = "patient_id"               # UBAH: kunci entitas ke tabel statis/riwayat
TARGET, ID = "target_class", "sample_id"
SEED, N_FOLD, N_REPEAT = 42, 5, 3
print("data:", FOLDER.resolve(), "| triase:", TRIASE, "| riwayat:", RIWAYAT_DIR, "| LightGBM:", HAS_LGB)

data: /home/claude/data | triase: data/Triase/Triase | riwayat: data/Riwayat Medis/Riwayat Medis | LightGBM: False


## 1. Tabel utama + waktu prediksi

In [2]:
train = pd.read_csv(FOLDER / "train.csv")
test = pd.read_csv(FOLDER / "test.csv")
base = pd.concat([train, test], ignore_index=True)       # fitur dibuat sekali untuk train+test
waktu = pd.read_csv(FOLDER / "index_encounters.csv")
waktu["prediction_time"] = pd.to_datetime(waktu["prediction_time"], utc=True)
base = base.merge(waktu[[KEY, "prediction_time"]], on=KEY, how="left")
assert base[KEY].is_unique and base["prediction_time"].notna().all()
print(base.shape); print(train[TARGET].value_counts())
print("pasien train ∩ test:", train[PASIEN].isin(test[PASIEN]).sum())   # 0 → KFold biasa aman

(5000, 5)
target_class
viral_sinusitis      1480
viral_pharyngitis     985
acute_bronchitis      940
strep_pharyngitis     195
Name: count, dtype: int64
pasien train ∩ test: 0


In [3]:
def norm(s):
    """teks → huruf kecil, simbol jadi spasi (menyatukan 'Cough / productive' dan 'cough productive')."""
    return s.astype(str).str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()


def angka(s):
    """'USD 1,234' / '$1,234' / '115.6k' / '28,02' → float. Koma = ribuan kalau diikuti 3 digit, selain itu desimal."""
    s = s.astype(str).str.lower().str.replace(r"usd|\$|\s", "", regex=True)
    kali = np.where(s.str.endswith("k"), 1000, 1)
    s = s.str.rstrip("k").str.replace(r",(?=\d{3}(\D|$))", "", regex=True).str.replace(",", ".")
    return pd.to_numeric(s, errors="coerce") * kali


assert angka(pd.Series(["USD 1,234", "$34,487", "115.6k", "28,02", "0,9", "12679.0", None])).round(2).tolist()[:6] == \
    [1234, 34487, 115600, 28.02, 0.9, 12679]

## 2. Tabel 1-ke-1: pasien

In [4]:
p = pd.read_csv(FOLDER / "patients.csv")
p["age"] = p["age_at_visit"].where(p["age_at_visit"].between(0, 120))   # 999 = kode kosong
p["income"] = angka(p["income_at_visit"])
p["marital"] = p["marital_at_visit"].str[0]                               # Married → M, Single → S
KATEGORI = ["gender", "race", "ethnicity", "marital", "county"]           # UBAH
feat = base.merge(p[[PASIEN, "age", "income"] + KATEGORI], on=PASIEN, how="left")
print(p[["age", "income"]].describe().round(1))

          age    income
count  4921.0    4545.0
mean     38.8  116145.9
std      21.4  168258.7
min       0.0       0.0
25%      20.5   32829.0
50%      38.3   68833.0
75%      56.0  124370.0
max      90.0  998248.0


## 3. Tabel long per sampel: vital → kolom

In [5]:
v = pd.read_csv(TRIASE / "triage_vitals.csv")
v["value"] = angka(v["value"])
print(v.groupby(["vital_name", "unit"])["value"].median())
KONVERSI = {"kPa": 7.50062, "fraction": 100}     # UBAH: satuan minoritas → satuan mayoritas
v["value"] *= v["unit"].map(KONVERSI).fillna(1)
vital = v.pivot_table(index=KEY, columns="vital_name", values="value", aggfunc="mean")
vital.columns = "vital_" + norm(pd.Series(vital.columns)).str.replace(" ", "_")
feat = feat.merge(vital, left_on=KEY, right_index=True, how="left")
print(v.groupby("vital_name")["value"].median())

vital_name        unit       
BMI               kg/m2           27.790
Diastolic BP      kPa             10.530
                  mmHg            79.000
Heart rate        /min            80.000
                  beats/min       80.000
Respiratory rate  /min            14.000
                  breaths/min     14.000
SpO2              %               94.000
                  fraction         0.940
Systolic BP       kPa             15.855
                  mmHg           118.000
Name: value, dtype: float64
vital_name
BMI                  27.79
Diastolic BP         79.00
Heart rate           80.00
Respiratory rate     14.00
SpO2                 94.00
Systolic BP         118.00
Name: value, dtype: float64


## 4. Gejala: satukan sinonim, lalu flag + skor keparahan

Kamus ini dibuat dari `sorted(norm(gejala).unique())`. Gejala di luar kamus masuk `lain` (tetap terbaca lewat TF-IDF).

In [6]:
SINONIM = {   # UBAH
    "throat": ["painful swallowing", "sore throat", "throat discomfort", "throat pain", "throat symptoms"],
    "cough": ["cough", "cough productive", "cough symptom", "coughing", "persistent cough"],
    "nasal": ["blocked nose", "nasal complaints", "nasal congestion", "nasal symptoms", "stuffy nose"],
    "fever": ["febrile feeling", "fever", "feverish", "raised temperature", "temperature complaint"],
    "systemic": ["body malaise", "feeling unwell", "general fatigue symptoms", "general malaise", "systemic symptoms"],
    "secretion": ["airway secretions", "mucus discharge", "mucus complaint", "respiratory secretions", "sputum production"],
    "dyspnea": ["breathing discomfort", "difficulty breathing", "dyspnea wheeze", "shortness of breath wheeze", "wheezy feeling"],
    "head": ["head and facial symptoms"],
}
KANON = {s: k for k, daftar in SINONIM.items() for s in daftar}

g = pd.read_csv(TRIASE / "triage_symptoms.csv")
g["nama"] = norm(g["symptom_concept"])
g["kanon"] = g["nama"].map(KANON).fillna("lain")
feat = feat.merge(pd.crosstab(g[KEY], g["kanon"]).add_prefix("ada_"), left_on=KEY, right_index=True, how="left")

sev = pd.read_csv(TRIASE / "triage_symptom_severity.csv")
sev["kanon"] = norm(sev["symptom_concept"]).map(KANON).fillna("lain")
per_kanon = sev.pivot_table(index=KEY, columns="kanon", values="severity_score", aggfunc="max").add_prefix("sev_")
ringkas = sev.groupby(KEY)["severity_score"].agg(sev_mean="mean", sev_max="max", sev_n="count")
feat = feat.merge(per_kanon, left_on=KEY, right_index=True, how="left").merge(ringkas, left_on=KEY, right_index=True, how="left")

keluhan = pd.read_csv(TRIASE / "presenting_complaints.csv").set_index(KEY)["presenting_complaints_text"]
feat["teks"] = feat[KEY].map(g.groupby(KEY)["nama"].agg(" | ".join)).fillna("") + " || " + norm(feat[KEY].map(keluhan).fillna(""))
print(g["kanon"].value_counts())

kanon
throat       3471
cough        2705
systemic     2571
nasal        2379
fever        2292
secretion    1977
head         1542
lain         1071
dyspnea      1057
Name: count, dtype: int64


## 5. Riwayat per pasien: **hanya sebelum waktu prediksi**

Tiap tabel → jumlah kejadian + jarak hari ke kejadian terakhir. Teks riwayat (nama penyakit/obat) sudah dicoba sebagai TF-IDF dan malah menurunkan CV, jadi tidak dipakai.

In [7]:
RIWAYAT = {   # UBAH: file → kolom tanggal
    "conditions": "recorded_at", "medications": "recorded_at", "procedures": "recorded_at",
    "immunizations": "recorded_at", "allergies": "recorded_at", "observations": "observed_at",
    "previous_encounter": "history_start",
}
for nama, kol in RIWAYAT.items():
    t = pd.read_csv(RIWAYAT_DIR / f"{nama}.csv")
    t["_t"] = pd.to_datetime(t[kol], utc=True, format="mixed")
    t = t.merge(base[[PASIEN, KEY, "prediction_time"]], on=PASIEN)
    sebelum = t["_t"] < t["prediction_time"]
    print(f"{nama:20s} baris {len(t):6d} | sesudah waktu prediksi (dibuang): {(~sebelum).sum()}")
    t = t[sebelum]
    feat[f"n_{nama}"] = feat[KEY].map(t.groupby(KEY).size()).fillna(0)
    feat[f"hari_{nama}"] = feat[KEY].map((t["prediction_time"] - t["_t"]).dt.days.groupby(t[KEY]).min())

conditions           baris   3583 | sesudah waktu prediksi (dibuang): 0
medications          baris   3467 | sesudah waktu prediksi (dibuang): 0
procedures           baris  17024 | sesudah waktu prediksi (dibuang): 0
immunizations        baris   1802 | sesudah waktu prediksi (dibuang): 0
allergies            baris      2 | sesudah waktu prediksi (dibuang): 0
observations         baris  55648 | sesudah waktu prediksi (dibuang): 0
previous_encounter   baris   4471 | sesudah waktu prediksi (dibuang): 0


## 6. Matriks fitur

In [8]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(feat["teks"])
X_teks = pd.DataFrame(tfidf.transform(feat["teks"]).toarray(), columns="tf_" + tfidf.get_feature_names_out())
for c in KATEGORI:
    feat[c] = feat[c].astype("category")
BUANG = [KEY, PASIEN, TARGET, ID, "prediction_time", "teks"]
X = pd.concat([feat.drop(columns=BUANG), X_teks], axis=1)
X.columns = X.columns.str.replace(r"[^A-Za-z0-9_]", "_", regex=True)   # LightGBM menolak simbol di nama kolom
konstan = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]      # mis. n_allergies (hanya 2 baris), token TF-IDF sangat langka
X = X.drop(columns=konstan); print("kolom konstan dibuang:", len(konstan), konstan[:6])

latih = feat[TARGET].notna().to_numpy()
KELAS = sorted(train[TARGET].unique())
y = feat.loc[latih, TARGET].map({k: i for i, k in enumerate(KELAS)}).to_numpy()
X_tr, X_te = X[latih].reset_index(drop=True), X[~latih].reset_index(drop=True)
print(X_tr.shape, X_te.shape)

kolom konstan dibuang: 1 ['hari_allergies']
(3600, 707) (1400, 707)


## 7. LightGBM + StratifiedKFold (OOF macro-F1)

In [9]:
PARAM = dict(n_estimators=3000, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8,
             subsample_freq=1, colsample_bytree=0.5, class_weight="balanced", random_state=SEED, verbose=-1)
# cadangan bila LightGBM tidak ada: konfigurasi yang dipilih lewat CV (probabilitas lebih terkalibrasi); ketidakseimbangan kelas
# ditangani di bagian 8, jadi tanpa class_weight
PARAM_HGB = dict(learning_rate=0.03, max_iter=250, max_leaf_nodes=8, min_samples_leaf=30, l2_regularization=3.0,
                 max_features=0.7, early_stopping=False, categorical_features="from_dtype")

def latih_fold(Xa, ya, Xb, yb, seed):
    if HAS_LGB:
        m = lgb.LGBMClassifier(**{**PARAM, "random_state": seed})
        m.fit(Xa, ya, eval_set=[(Xb, yb)], callbacks=[lgb.early_stopping(100, verbose=False)])
        return m, m.best_iteration_
    return HistGradientBoostingClassifier(**PARAM_HGB, random_state=seed).fit(Xa, ya), PARAM_HGB["max_iter"]

oof = np.zeros((len(y), len(KELAS)))
pred_test = np.zeros((len(X_te), len(KELAS)))
penting = pd.Series(0.0, index=X.columns)
for r in range(N_REPEAT):
    for f, (a, b) in enumerate(StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED + r).split(X_tr, y)):
        m, it = latih_fold(X_tr.iloc[a], y[a], X_tr.iloc[b], y[b], SEED + r)
        oof[b] += m.predict_proba(X_tr.iloc[b]) / N_REPEAT
        pred_test += m.predict_proba(X_te) / (N_FOLD * N_REPEAT)
        if HAS_LGB:
            penting += m.feature_importances_ / (N_FOLD * N_REPEAT)
        pb = m.predict_proba(X_tr.iloc[b])
        print(f"ulangan {r} fold {f}: iter {it}, macro-F1 {f1_score(y[b], pb.argmax(1), average='macro'):.4f}")
print("OOF macro-F1 (rata-rata ulangan):", round(f1_score(y, oof.argmax(1), average="macro"), 4))
print(classification_report(y, oof.argmax(1), target_names=KELAS, digits=3))
if HAS_LGB:
    print(penting.sort_values(ascending=False).head(20).round(0))

ulangan 0 fold 0: iter 250, macro-F1 0.6872
ulangan 0 fold 1: iter 250, macro-F1 0.6936
ulangan 0 fold 2: iter 250, macro-F1 0.7631
ulangan 0 fold 3: iter 250, macro-F1 0.7273
ulangan 0 fold 4: iter 250, macro-F1 0.7001
ulangan 1 fold 0: iter 250, macro-F1 0.7084
ulangan 1 fold 1: iter 250, macro-F1 0.7083
ulangan 1 fold 2: iter 250, macro-F1 0.7188
ulangan 1 fold 3: iter 250, macro-F1 0.6952
ulangan 1 fold 4: iter 250, macro-F1 0.6985
ulangan 2 fold 0: iter 250, macro-F1 0.7079
ulangan 2 fold 1: iter 250, macro-F1 0.7147
ulangan 2 fold 2: iter 250, macro-F1 0.7352
ulangan 2 fold 3: iter 250, macro-F1 0.7255
ulangan 2 fold 4: iter 250, macro-F1 0.6719
OOF macro-F1 (rata-rata ulangan): 0.712
                   precision    recall  f1-score   support

 acute_bronchitis      0.907     0.862     0.884       940
strep_pharyngitis      0.463     0.190     0.269       195
viral_pharyngitis      0.741     0.808     0.773       985
  viral_sinusitis      0.900     0.945     0.922      1480

   

## 8. Bobot kelas untuk macro-F1

Kalikan probabilitas tiap kelas dengan bobot, cari bobot terbaik di OOF (koordinat demi koordinat). Kalau naiknya cuma ~0,001, abaikan.

In [10]:
MIN_GAIN = 0.001                       # kenaikan <= ini dianggap noise -> bobot diabaikan
KANDIDAT = np.round(np.arange(0.5, 2.51, 0.05), 2)

def f1_bobot(yy, PP, w):
    return f1_score(yy, (PP * w).argmax(1), average="macro")

def cari_bobot(yy, PP, passes=3):
    """Koordinat demi koordinat: ubah bobot satu kelas, ambil yang memaksimalkan macro-F1."""
    w = np.ones(PP.shape[1])
    for _ in range(passes):
        for k in range(len(w)):
            skor = [f1_bobot(yy, PP, np.where(np.arange(len(w)) == k, c, w)) for c in KANDIDAT]
            w[k] = KANDIDAT[int(np.argmax(skor))]
    return w

w = cari_bobot(y, oof)
f1_awal, f1_akhir = f1_bobot(y, oof, np.ones(len(KELAS))), f1_bobot(y, oof, w)
gain = f1_akhir - f1_awal
w_pakai = w if gain > MIN_GAIN else np.ones(len(KELAS))
print("bobot:", dict(zip(KELAS, w.round(2))))
print(f"macro-F1 OOF: {f1_awal:.4f} -> {f1_akhir:.4f} (gain {gain:+.4f}) | dipakai: {'ya' if gain > MIN_GAIN else 'TIDAK (gain terlalu kecil)'}")

# uji kestabilan: bobot disetel di separuh data, dinilai di separuh lain (gain harus tetap positif)
rng, gains = np.random.RandomState(0), []
for _ in range(10):
    idx = rng.permutation(len(y)); i1, i2 = idx[: len(y) // 2], idx[len(y) // 2:]
    w1 = cari_bobot(y[i1], oof[i1])
    gains.append(f1_bobot(y[i2], oof[i2], w1) - f1_bobot(y[i2], oof[i2], np.ones(len(KELAS))))
print(f"gain pada data yang tidak dipakai menyetel: rata-rata {np.mean(gains):+.4f}, terendah {np.min(gains):+.4f}")

bobot: {'acute_bronchitis': np.float64(0.85), 'strep_pharyngitis': np.float64(1.7), 'viral_pharyngitis': np.float64(1.0), 'viral_sinusitis': np.float64(0.5)}
macro-F1 OOF: 0.7120 -> 0.7412 (gain +0.0292) | dipakai: ya
gain pada data yang tidak dipakai menyetel: rata-rata +0.0211, terendah +0.0077


## 9. Submission

In [11]:
contoh = pd.read_csv(FOLDER / "sample_submission.csv")
hasil = pd.DataFrame({ID: test[ID], TARGET: np.array(KELAS)[(pred_test * w_pakai).argmax(1)]})
sub = contoh[[ID]].merge(hasil, on=ID, how="left")
assert len(sub) == len(contoh) and sub[TARGET].notna().all(), "ID test tidak cocok dengan sample_submission"
sub.to_csv(OUT / "submission.csv", index=False)
pd.DataFrame(oof, columns=KELAS).assign(**{ID: train[ID].to_numpy()}).to_csv(OUT / "oof_model.csv", index=False)
print(sub[TARGET].value_counts(normalize=True).round(3)); print("tersimpan:", OUT / "submission.csv")

target_class
viral_sinusitis      0.368
viral_pharyngitis    0.326
acute_bronchitis     0.255
strep_pharyngitis    0.051
Name: proportion, dtype: float64
tersimpan: /home/claude/../outputs/submission.csv
